# OWL Adapter Tutorial

The **OWL adapter** reads an OWL file directly, using
[py-horned-owl](https://github.com/ontology-tools/py-horned-owl) as the parser and object
model. Unlike adapters that work over a normalized projection of an ontology (such as the
[SQL adapter](https://incatools.github.io/ontology-access-kit/implementations/sqldb.html)
over a [Semantic SQL](https://github.com/INCATools/semantic-sql) database), this adapter
keeps the OWL axioms themselves in memory. That makes it the right choice when you need to:

- work with an ontology *as it is on disk*, with no pre-processing step
- query and edit at the level of **OWL axioms**, not just entities and relationships
- load `.ofn`, `.owl`, `.owx`, `.omn` or `.ttl` files with no conversion

This tutorial uses the small GO subset that ships with the OAK test suite. Everything shown
here works the same way on a full-size ontology.

### Change directory so that test files are directly accessible

This notebook lives in a subfolder of the OAK repository, so we move to the repository root
first. If you are following along with your own ontology you can skip this.

In [1]:
%cd ../../../..

/home/user/ontology-access-kit


In [2]:
!ls tests/input/go-nucleus.*

tests/input/go-nucleus.cx      tests/input/go-nucleus.obo
tests/input/go-nucleus.db      tests/input/go-nucleus.ofn
tests/input/go-nucleus.ic.tsv  tests/input/go-nucleus.owl
tests/input/go-nucleus.json    tests/input/go-nucleus.owl.ttl


The test ontology declares a few relations twice, once with an underscore and once without
(`part_of` and `part of`). OAK warns when it has to pick one; we quieten those warnings here
so the rest of the notebook reads cleanly.

In [3]:
import logging
import warnings

logging.getLogger("oaklib").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
%env PYTHONWARNINGS=ignore

env: PYTHONWARNINGS=ignore


## Loading an ontology

The adapter is selected from the file suffix, so no explicit selector is needed for the
common OWL syntaxes. `go-nucleus.ofn` is OWL functional syntax; `go-nucleus.owl` is
RDF/XML. Both load into the same object model.

In [4]:
from oaklib import get_adapter

adapter = get_adapter("tests/input/go-nucleus.ofn")
print(type(adapter).__name__)

FunOwlImplementation


An ontology declares its own IRI, and any ontology-level annotations are available too:

In [5]:
for ontology in adapter.ontologies():
    print(ontology)
    for predicate, values in adapter.ontology_metadata_map(ontology).items():
        print(f"  {predicate} = {values}")

obo:go.owl
  oio:hasOBOFormatVersion = ['1.2']


## Basic lookups

The core of OAK is the
[BasicOntologyInterface](https://incatools.github.io/ontology-access-kit/interfaces/basic-ontology-interface.html):
labels, definitions, synonyms and metadata, keyed by CURIE.

In [6]:
NUCLEUS = "GO:0005634"

print(adapter.label(NUCLEUS))
print(adapter.definition(NUCLEUS))

nucleus
A membrane-bounded organelle of eukaryotic cells in which chromosomes are housed and replicated. In most cells, the nucleus contains all of the cell's chromosomes except the organellar chromosomes, and is the site of RNA synthesis and processing. In some species, or in specialized cell types, RNA metabolism or DNA replication may be absent.


Definitions in OBO ontologies carry provenance as *annotations on the annotation axiom*.
Pass `include_metadata=True` to get it:

In [7]:
for curie, definition, metadata in adapter.definitions([NUCLEUS], include_metadata=True):
    print(curie)
    print(f"  definition: {definition[:60]}...")
    print(f"  provenance: {metadata}")

GO:0005634
  definition: A membrane-bounded organelle of eukaryotic cells in which ch...
  provenance: {'oio:hasDbXref': ['GOC:go_curators']}


`labels` works over any iterable, and lookups run in the other direction too:

In [8]:
for curie, label in adapter.labels(["GO:0005634", "GO:0005773", "BFO:0000050"]):
    print(f"{curie}\t{label}")

print()
print(adapter.curies_by_label("vacuole"))

GO:0005634	nucleus
GO:0005773	vacuole
BFO:0000050	part of

['GO:0005773']


## Synonyms, with scope and provenance

`entity_aliases` gives you every name for a term. If you need the *scope* of each synonym
(exact, narrow, broad, related), its xrefs, or its ontology-specific synonym *type*, use
`synonym_property_values`, which returns
[OBO Graph](https://github.com/geneontology/obographs) `SynonymPropertyValue` objects.

In [9]:
sorted(adapter.entity_aliases(NUCLEUS))

['cell nucleus', 'horsetail nucleus', 'nucleus']

In [10]:
for _, synonym in sorted(adapter.synonym_property_values([NUCLEUS]), key=lambda s: s[1].val):
    print(f"{synonym.pred:20s} {synonym.val!r}")
    print(f"  type:  {synonym.synonymType}")
    print(f"  xrefs: {sorted(synonym.xrefs)}")

hasExactSynonym      'cell nucleus'
  type:  systematic_synonym
  xrefs: []
hasNarrowSynonym     'horsetail nucleus'
  type:  None
  xrefs: ['GOC:al', 'GOC:mah', 'GOC:vw', 'PMID:15030757']


## Metadata, xrefs and subsets

`entity_metadata_map` returns every annotation assertion about an entity, keyed by
predicate. Note that `owl:deprecated` comes back as a Python boolean.

In [11]:
metadata = adapter.entity_metadata_map(NUCLEUS)
for predicate, values in sorted(metadata.items()):
    print(f"{predicate:25s} {sorted(values)}")

IAO:0000115               ["A membrane-bounded organelle of eukaryotic cells in which chromosomes are housed and replicated. In most cells, the nucleus contains all of the cell's chromosomes except the organellar chromosomes, and is the site of RNA synthesis and processing. In some species, or in specialized cell types, RNA metabolism or DNA replication may be absent."]
RO:0002161                ['NCBITaxon:2']
oio:hasDbXref             ['NIF_Subcellular:sao1702920020', 'Wikipedia:Cell_nucleus']
oio:hasExactSynonym       ['cell nucleus']
oio:hasNarrowSynonym      ['horsetail nucleus']
oio:hasOBONamespace       ['cellular_component']
oio:id                    ['GO:0005634']
oio:inSubset              ['obo:go#goslim_agr', 'obo:go#goslim_aspergillus', 'obo:go#goslim_candida', 'obo:go#goslim_chembl', 'obo:go#goslim_drosophila', 'obo:go#goslim_flybase_ribbon', 'obo:go#goslim_generic', 'obo:go#goslim_metagenomics', 'obo:go#goslim_mouse', 'obo:go#goslim_pir', 'obo:go#goslim_plant', 'obo:go#go

Subsets ("slims") are first-class. `subsets()` lists them, and membership can be queried in
either direction:

In [12]:
print(sorted(adapter.subsets())[:6])
print()
print(sorted(adapter.subset_members("goslim_generic")))
print()
print(sorted(subset for _, subset in adapter.terms_subsets([NUCLEUS])))

['chebi_ph7_3', 'gocheck_do_not_annotate', 'gocheck_do_not_manually_annotate', 'goslim_agr', 'goslim_aspergillus', 'goslim_candida']

['GO:0003674', 'GO:0005575', 'GO:0005622', 'GO:0005634', 'GO:0005635', 'GO:0005737', 'GO:0005773', 'GO:0005886', 'GO:0008150', 'GO:0009579', 'GO:0015979', 'GO:0016301', 'GO:0043226']

['goslim_agr', 'goslim_aspergillus', 'goslim_candida', 'goslim_chembl', 'goslim_drosophila', 'goslim_flybase_ribbon', 'goslim_generic', 'goslim_metagenomics', 'goslim_mouse', 'goslim_pir', 'goslim_plant', 'goslim_yeast']


Cross-references are exposed both as raw pairs and as
[SSSOM](https://mapping-commons.github.io/sssom/) mappings:

In [13]:
for mapping in sorted(adapter.get_sssom_mappings_by_curie(NUCLEUS), key=lambda m: m.object_id):
    print(f"{mapping.subject_id} {mapping.predicate_id} {mapping.object_id}")

GO:0005634 oio:hasDbXref NIF_Subcellular:sao1702920020
GO:0005634 oio:hasDbXref Wikipedia:Cell_nucleus


## Relationships and graph traversal

`relationships` projects OWL axioms into subject-predicate-object triples: `SubClassOf`
axioms become `rdfs:subClassOf` edges, and existential restrictions
(`SubClassOf(C ObjectSomeValuesFrom(R D))`) become `C R D` edges.

In [14]:
for s, p, o in sorted(adapter.relationships(subjects=["GO:0031965"])):
    print(f"{s} {adapter.label(s)!r}  --{adapter.label(p) or p}-->  {o} {adapter.label(o)!r}")

GO:0031965 'nuclear membrane'  --part of-->  GO:0005634 'nucleus'
GO:0031965 'nuclear membrane'  --part of-->  GO:0005635 'nuclear envelope'
GO:0031965 'nuclear membrane'  --rdfs:subClassOf-->  GO:0031090 'organelle membrane'


Passing `include_entailed=True` returns the *deductive closure* over the class hierarchy,
sub-properties and transitive properties -- computed in memory, with no reasoner required.
Here nuclear membrane is entailed to be part of everything the nuclear envelope is part of:

In [15]:
from oaklib.datamodels.vocabulary import IS_A, PART_OF

direct = {o for _, _, o in adapter.relationships(subjects=["GO:0031965"], predicates=[PART_OF])}
entailed = {
    o
    for _, _, o in adapter.relationships(
        subjects=["GO:0031965"], predicates=[PART_OF], include_entailed=True
    )
}
print(f"direct part_of parents:   {sorted(direct)}")
print(f"entailed part_of ancestors: {len(entailed)} terms")
print(sorted(entailed)[:8])

direct part_of parents:   ['GO:0005634', 'GO:0005635']
entailed part_of ancestors: 20 terms
['BFO:0000002', 'BFO:0000004', 'BFO:0000040', 'CARO:0000000', 'CARO:0000003', 'CARO:0000006', 'CARO:0030000', 'CL:0000000']


`ancestors` and `descendants` walk the graph over whichever predicates you choose:

In [16]:
isa_only = set(adapter.ancestors(NUCLEUS, predicates=[IS_A], reflexive=False))
isa_and_part_of = set(adapter.ancestors(NUCLEUS, predicates=[IS_A, PART_OF], reflexive=False))

print(f"is_a ancestors:            {len(isa_only)}")
print(f"is_a + part_of ancestors:  {len(isa_and_part_of)}")
for curie in sorted(isa_and_part_of - isa_only):
    print(f"  only via part_of: {curie} ! {adapter.label(curie)}")

is_a ancestors:            11
is_a + part_of ancestors:  15
  only via part_of: CARO:0000003 ! connected anatomical structure
  only via part_of: CARO:0000006 ! material anatomical entity
  only via part_of: CL:0000000 ! cell
  only via part_of: GO:0005622 ! intracellular anatomical structure


## Logical definitions

Equivalence axioms of the genus-differentia form are reported as OBO Graph logical
definition axioms:

In [17]:
for ldef in adapter.logical_definitions([NUCLEUS, "GO:0031965"]):
    print(f"{ldef.definedClassId} ! {adapter.label(ldef.definedClassId)}")
    for genus in ldef.genusIds:
        print(f"  genus:       {genus} ! {adapter.label(genus)}")
    for restriction in ldef.restrictions:
        print(
            f"  restriction: {restriction.propertyId} ! {adapter.label(restriction.propertyId)}"
            f" -> {restriction.fillerId} ! {adapter.label(restriction.fillerId)}"
        )

GO:0031965 ! nuclear membrane
  genus:       GO:0016020 ! membrane
  restriction: BFO:0000050 ! part of -> GO:0005634 ! nucleus


## Working at the axiom level

This is what the OWL adapter gives you that a triple- or edge-oriented adapter cannot: the
OWL axioms themselves. `axioms()` yields py-horned-owl objects, and `filter_axioms` selects
them by type or by the entities they mention.

In [18]:
from collections import Counter

Counter(type(axiom).__name__ for axiom in adapter.axioms()).most_common(8)

[('AnnotationAssertion', 1678),
 ('SubClassOf', 336),
 ('DeclareClass', 204),
 ('SubObjectPropertyOf', 152),
 ('DeclareObjectProperty', 95),
 ('EquivalentClasses', 63),
 ('DeclareAnnotationProperty', 25),
 ('ObjectPropertyRange', 23)]

In [19]:
from oaklib.interfaces.owl_interface import AxiomFilter
from pyhornedowl.model import SubClassOf

subclass_axioms = list(adapter.filter_axioms(AxiomFilter(type=SubClassOf, about=NUCLEUS)))
for axiom in subclass_axioms:
    print(axiom)

SubClassOf(<http://purl.obolibrary.org/obo/GO_0005634> ObjectSomeValuesFrom(<http://purl.obolibrary.org/obo/RO_0002162> <http://purl.obolibrary.org/obo/NCBITaxon_2759>))
SubClassOf(<http://purl.obolibrary.org/obo/GO_0005634> ObjectSomeValuesFrom(<http://purl.obolibrary.org/obo/RO_0002160> <http://purl.obolibrary.org/obo/NCBITaxon_2759>))
SubClassOf(<http://purl.obolibrary.org/obo/GO_0005634> <http://purl.obolibrary.org/obo/GO_0043231>)


Disjointness, transitive properties and property chains are all queryable. `is_disjoint`
takes the class hierarchy into account, so it holds for entailed pairs as well as asserted
ones:

In [20]:
for left, right in sorted(adapter.disjoint_pairs())[:5]:
    print(f"{left} ! {adapter.label(left)}   DisjointWith   {right} ! {adapter.label(right)}")

print()
print("nucleus vs cytoplasm (asserted): ", adapter.is_disjoint("GO:0005634", "GO:0005737"))
print("nucleus vs biological process:   ", adapter.is_disjoint("GO:0005634", "GO:0008150"))
print("nucleus vs nuclear membrane:     ", adapter.is_disjoint("GO:0005634", "GO:0031965"))

BFO:0000002 ! continuant   DisjointWith   BFO:0000003 ! occurrent
BFO:0000004 ! independent continuant   DisjointWith   BFO:0000020 ! specifically dependent continuant
CL:0000000 ! cell   DisjointWith   GO:0043226 ! organelle
GO:0003674 ! molecular_function   DisjointWith   GO:0005575 ! cellular_component
GO:0003674 ! molecular_function   DisjointWith   GO:0008150 ! biological_process

nucleus vs cytoplasm (asserted):  True
nucleus vs biological process:    True
nucleus vs nuclear membrane:      False


In [21]:
print("transitive object properties:")
for prop in sorted(adapter.transitive_object_properties())[:5]:
    print(f"  {prop} ! {adapter.label(prop)}")

print()
print("property chains:")
for prop, chain in list(adapter.simple_subproperty_of_chains())[:3]:
    labels = " o ".join(adapter.label(link) or link for link in chain)
    print(f"  {labels}  ->  {adapter.label(prop) or prop}")

transitive object properties:
  BFO:0000050 ! part of
  BFO:0000051 ! has_part
  BFO:0000062 ! preceded_by
  BFO:0000063 ! precedes
  RO:0002086 ! ends after

property chains:
  enables o causally upstream of or within, negative effect  ->  acts upstream of or within, negative effect
  inheres in part of o part of  ->  inheres in part of
  involved in o regulates  ->  involved in regulation of


## Projecting to OBO Graphs and other formats

The adapter implements the OBO Graph interface, so any term can be projected to a node with
its full metadata, and the whole ontology (or a subgraph) can be written out in other
formats.

In [22]:
node = adapter.node(NUCLEUS, include_metadata=True)
print(node.id, "|", node.lbl, "|", node.type)
print("definition:", node.meta.definition.val[:50], "...", node.meta.definition.xrefs)
print("xrefs:     ", [xref.val for xref in node.meta.xrefs])
print("subsets:   ", sorted(node.meta.subsets)[:3])
print("synonyms:  ", sorted((s.pred, s.val) for s in node.meta.synonyms))

GO:0005634 | nucleus | CLASS
definition: A membrane-bounded organelle of eukaryotic cells i ... ['GOC:go_curators']
xrefs:      ['Wikipedia:Cell_nucleus', 'NIF_Subcellular:sao1702920020']
subsets:    ['obo:go#goslim_agr', 'obo:go#goslim_aspergillus', 'obo:go#goslim_candida']
synonyms:   [('hasExactSynonym', 'cell nucleus'), ('hasNarrowSynonym', 'horsetail nucleus')]


`dump` writes OWL syntaxes natively via py-horned-owl, and routes other formats
(OBO Graph JSON, obo format, FHIR, CX) through the OBO Graph projection:

In [23]:
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    out = Path(tmpdir) / "go-nucleus.json"
    adapter.dump(str(out), "json")
    graph = json.loads(out.read_text())["graphs"][0]
    print(f"{len(graph['nodes'])} nodes, {len(graph['edges'])} edges")
    print(f"{len(graph.get('logicalDefinitionAxioms', []))} logical definition axioms")

295 nodes, 532 edges
49 logical definition axioms


## Search and text annotation

Search runs over labels, synonyms, identifiers and xrefs, and supports partial, regex and
starts-with matching:

In [24]:
from oaklib.datamodels.search import SearchConfiguration

print(list(adapter.basic_search("nucleus")))
print(sorted(adapter.basic_search("nucl", config=SearchConfiguration(is_partial=True))))

['GO:0005634']
['CHEBI:33252', 'CHEBI:33253', 'CHEBI:36347', 'GO:0005622', 'GO:0005634', 'GO:0005635', 'GO:0031965', 'PATO:0001404', 'PATO:0001405', 'PATO:0001406', 'PATO:0001407', 'PATO:0001908', 'PATO:0002505']


Because the adapter can search, it can also annotate free text -- useful for a first pass at
recognising ontology terms in a document:

In [25]:
text = "The nucleus is the part of the cell that contains the DNA."
for annotation in sorted(adapter.annotate_text(text), key=lambda a: a.subject_start):
    matched = text[annotation.subject_start - 1 : annotation.subject_end]
    print(f"{annotation.object_id:15s} {annotation.object_label!r:25s} matched {matched!r}")

GO:0005634      'nucleus'                 matched 'nucleus'
BFO:0000050     'part of'                 matched 'part of'
CL:0000000      'cell'                    matched 'cell'


## Editing an ontology

The OWL adapter is writable. Changes are expressed in
[KGCL](https://github.com/INCATools/kgcl), applied to the in-memory axioms, and can then be
written back out. Caches (labels, relationships, entailments) are invalidated automatically.

In [26]:
from kgcl_schema.datamodel import kgcl
from oaklib.utilities.kgcl_utilities import generate_change_id

editable = get_adapter("tests/input/go-nucleus.ofn")
print("before:", editable.label("GO:0005773"))

editable.apply_patch(
    kgcl.NodeRename(
        id=generate_change_id(), about_node="GO:0005773", new_value="the vacuole"
    )
)
print("after: ", editable.label("GO:0005773"))
print("lookup by new label:", editable.curies_by_label("the vacuole"))

before: vacuole
after:  the vacuole


lookup by new label: ['GO:0005773']


## Summary statistics

Summary statistics work over the OWL file directly:

In [27]:
stats = adapter.branch_summary_statistics()
print(f"classes:                {stats.class_count}")
print(f"  with text definitions: {stats.class_count_with_text_definitions}")
print(f"is_a edges:             {stats.edge_count_by_predicate[IS_A].filtered_count}")
print(f"part_of edges:          {stats.edge_count_by_predicate[PART_OF].filtered_count}")
print(f"distinct synonyms:      {stats.distinct_synonym_count}")

classes:                204
  with text definitions: 98
is_a edges:             221
part_of edges:          29
distinct synonyms:      260


## Performance notes

Everything is held in memory, so after the initial parse, lookups are fast. Annotation
assertions are indexed by subject the first time any metadata is requested, which keeps bulk
operations linear in the size of the ontology rather than quadratic.

In [28]:
import time

def timeit(label, fn):
    start = time.time()
    result = fn()
    size = f"n={len(result)}" if isinstance(result, list) else ""
    print(f"{label:26s} {time.time() - start:6.3f}s  {size}")
    return result

owl = timeit("load OWL file", lambda: get_adapter("tests/input/go-nucleus.ofn"))
owl_entities = timeit("entities()", lambda: list(owl.entities()))
_ = timeit("labels(all entities)", lambda: list(owl.labels(owl_entities)))
_ = timeit("relationships()", lambda: list(owl.relationships()))
_ = timeit("entailed relationships", lambda: list(owl.relationships(include_entailed=True)))

load OWL file               0.027s  
entities()                  0.039s  n=295
labels(all entities)        0.000s  n=295
relationships()             0.010s  n=532


entailed relationships      0.243s  n=6747


For comparison, the same queries against the Semantic SQL build of the same ontology. The
SQL adapter wins on start-up for large files (nothing is parsed up front) and is the better
choice when you want pre-computed entailments; the OWL adapter wins when you need the axioms
themselves, or when the ontology has no pre-built database.

In [29]:
sql = timeit("load SQLite database", lambda: get_adapter("tests/input/go-nucleus.db"))
sql_entities = timeit("entities()", lambda: list(sql.entities()))
_ = timeit("labels(all entities)", lambda: list(sql.labels(sql_entities)))
_ = timeit("relationships()", lambda: list(sql.relationships()))

load SQLite database        0.008s  
entities()                  0.027s  n=305
labels(all entities)        0.007s  n=346
relationships()             0.018s  n=527


## Command line

Every operation above is also available from the command line, using the same selector:

In [30]:
!runoak -i tests/input/go-nucleus.ofn --quiet info GO:0005634

GO:0005634 ! nucleus


In [31]:
!runoak -i tests/input/go-nucleus.ofn --quiet relationships GO:0031965

## Further reading

- [OWL adapter API documentation](https://incatools.github.io/ontology-access-kit/implementations/funowl.html)
- [OwlInterface](https://incatools.github.io/ontology-access-kit/interfaces/owl-interface.html) -- axiom-level access
- [Selectors](https://incatools.github.io/ontology-access-kit/packages/selectors.html) -- how input strings map to adapters